# Alfido Tech — Sales Performance Analysis

## Project objective
Analyze sales performance across regions, categories, products, and time; identify best- and low-performing products; explore seasonality; and recommend five tactical improvements.

## Dataset
The supplied archive contains `superstore_final_dataset (1).csv` (9,800 rows and 18 columns). The notebook reads the included UTF-8 normalized copy from `data/superstore_sales_data.csv` when run from the project root.

**Important scope limitation:** the supplied file contains a `Sales` field but no `Profit`, `Cost`, `Discount`, website sessions, leads, or conversion events. Therefore, revenue and average order value can be calculated, but profit margin and conversion metrics cannot be calculated from this dataset. Low sales should not be interpreted as low profit or poor profitability.

## Tools
Python, Pandas, NumPy, Matplotlib, and Seaborn.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Run from the project root. If needed, update this relative path.
df = pd.read_csv("data/superstore_sales_data.csv", parse_dates=["Order_Date", "Ship_Date"])
df.head()


## 1. Data quality and preparation
Check dimensions, data types, missing values, duplicate rows, date range, and key uniqueness. Postal code has missing values; it is not required for the regional or product analysis, so it is retained as missing rather than imputed.

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
print("Exact duplicate rows:", df.duplicated().sum())
print("Unique orders:", df["Order_ID"].nunique())
print("Unique customers:", df["Customer_ID"].nunique())
print("Order date range:", df["Order_Date"].min(), "to", df["Order_Date"].max())

# Ensure numeric sales and parsed dates; remove rows missing core analysis fields.
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
df = df.dropna(subset=["Order_ID", "Order_Date", "Sales"]).copy()
df["Month"] = df["Order_Date"].dt.to_period("M").dt.to_timestamp()
df["Year"] = df["Order_Date"].dt.year
df["Month_Number"] = df["Order_Date"].dt.month


## 2. KPI calculations
Revenue is the sum of Sales. Average order value (AOV) is total sales divided by the number of distinct orders. This dataset does not support profit margin or conversion rate calculations.

In [ ]:
total_revenue = df["Sales"].sum()
total_orders = df["Order_ID"].nunique()
unique_customers = df["Customer_ID"].nunique()
aov = total_revenue / total_orders

kpis = pd.Series({
    "Revenue": total_revenue,
    "Orders": total_orders,
    "Average Order Value": aov,
    "Unique Customers": unique_customers
})
display(kpis.to_frame("value").round(2))


## 3. Regional performance
Compare total sales and each region's share of revenue.

In [ ]:
region_sales = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
region_summary = region_sales.to_frame("revenue")
region_summary["revenue_share_pct"] = region_summary["revenue"] / total_revenue * 100
display(region_summary.round(2))

region_sales.sort_values().plot(kind="barh", title="Revenue by Region", figsize=(9, 5))
plt.xlabel("Sales")
plt.tight_layout()
plt.show()


## 4. Category and sub-category performance
Compare sales by category and sub-category. These are sales rankings only; profitability cannot be inferred.

In [ ]:
category_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
subcategory_sales = df.groupby("Sub_Category")["Sales"].sum().sort_values(ascending=False)
display(category_sales.to_frame("revenue").round(2))
display(subcategory_sales.to_frame("revenue").round(2))

category_sales.sort_values().plot(kind="barh", title="Revenue by Category", figsize=(8, 4))
plt.xlabel("Sales")
plt.tight_layout()
plt.show()


## 5. Product performance
Aggregate sales by product. The top products have the highest recorded sales; the bottom products have the lowest recorded sales in this extract. A low sales total can reflect low demand, limited availability, or a niche product; it does not prove the product is unprofitable.

In [ ]:
product_sales = (df.groupby(["Product_ID", "Product_Name"], as_index=False)["Sales"]
                 .sum().sort_values("Sales", ascending=False))
display(product_sales.head(10))
display(product_sales.tail(10).sort_values("Sales"))

product_sales.head(10).set_index("Product_Name")["Sales"].sort_values().plot(
    kind="barh", title="Top 10 Products by Revenue", figsize=(10, 6))
plt.xlabel("Sales")
plt.tight_layout()
plt.show()


## 6. Time trends and seasonality
Monthly revenue shows the trend through time. Calendar-month averages pool the observed years and can help identify recurring patterns, but interpretation should consider the limited four-year period and transaction-line nature of the data.

In [ ]:
monthly_revenue = df.groupby("Month")["Sales"].sum()
monthly_orders = df.groupby("Month")["Order_ID"].nunique()
monthly_revenue.plot(title="Monthly Revenue Trend", figsize=(12, 5))
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

calendar_month_avg = df.groupby("Month_Number")["Sales"].mean()
calendar_month_avg.plot(kind="bar", title="Average Transaction-Line Sales by Calendar Month", figsize=(9, 4))
plt.xlabel("Month number")
plt.ylabel("Average sales per row")
plt.tight_layout()
plt.show()

display(df.groupby("Year")["Sales"].sum().to_frame("annual_revenue").round(2))


## 7. Shipping performance (supporting operational view)
Calculate the number of days between order and ship date. This is a shipping lead-time indicator, not a delivery-time measure because the dataset does not contain delivery dates.

In [ ]:
df["Ship_Lead_Days"] = (df["Ship_Date"] - df["Order_Date"]).dt.days
display(df["Ship_Lead_Days"].describe().to_frame("ship_lead_days"))
display(df.groupby("Ship_Mode")["Ship_Lead_Days"].median().sort_values().to_frame("median_ship_lead_days"))


## 8. Five tactical recommendations for Alfido Tech

1. **Prioritize high-revenue regions:** protect availability and sales coverage in the West and East, which lead this dataset. Test targeted campaigns in Central and South after diagnosing local product mix and customer needs.
2. **Build category-specific campaigns:** Technology is the largest category by sales. Use bundles and cross-sell recommendations, while testing category-level campaigns for Furniture and Office Supplies.
3. **Review low-sales products before delisting:** investigate the bottom-selling products for stock availability, listing quality, pricing, and demand. Do not treat low sales as proof of low profit; obtain product cost and margin data first.
4. **Plan inventory and promotions around observed monthly patterns:** use the monthly trend and month-of-year comparison to plan stock, staffing, and campaign calendars. Validate the pattern with additional years before committing large budgets.
5. **Instrument the sales funnel and profitability:** capture sessions/leads, cart additions, orders, returns, discounts, unit costs, and profit. This will enable conversion rate, gross margin, return rate, and campaign ROI analysis in the next iteration.

## 9. Limitations and next data requests
- Profit and cost are absent: profit, gross margin, and profit margin cannot be computed.
- Traffic, leads, and funnel events are absent: conversion rate cannot be computed.
- Returns, discounts, and inventory availability are absent: product performance cannot be adjusted for returns, discounting, or stockouts.
- Sales is treated as revenue for this analysis; confirm the business definition (gross/net, tax, returns) with the data owner.
